# YouTube Playlist Downloader

Downloads a playlist into an organized folder while preserving playlist order. Use this only for content you own or have permission to download.

Run each cell from top to bottom.

## 1. Install the downloader

After this finishes, restart the notebook kernel only if your notebook asks you to. `ffmpeg` is also required for merging high-quality video/audio and creating MP3 files.

In [ ]:
%pip install -q --upgrade yt-dlp

### Optional: install ffmpeg in Google Colab

Uncomment and run this cell only when using Google Colab. For a local notebook, install ffmpeg using your operating system's package manager.

In [ ]:
# !apt-get update -qq && apt-get install -y -qq ffmpeg

## 2. Choose your settings

Paste your playlist URL below. The default downloads video up to 1080p.

In [ ]:
from pathlib import Path

PLAYLIST_URL = "PASTE_YOUR_PLAYLIST_URL_HERE"
OUTPUT_FOLDER = Path("downloads")

AUDIO_ONLY = False       # True creates MP3 files
MAX_RESOLUTION = 1080    # Try 720 for smaller files
DOWNLOAD_SUBTITLES = False

# Optional playlist range. Use None to download everything.
START_ITEM = None        # Example: 10
END_ITEM = None          # Example: 20

# Optional: use browser cookies for content your account can access.
# Examples: "chrome", "firefox", "edge", or None.
COOKIES_FROM_BROWSER = None

## 3. Download the playlist

Files are saved as `Playlist Name/001 - Video Title [video-id].mp4`. Running the cell again skips completed videos and resumes partial downloads.

In [ ]:
import shutil
import yt_dlp

if PLAYLIST_URL == "PASTE_YOUR_PLAYLIST_URL_HERE" or not PLAYLIST_URL.strip():
    raise ValueError("Paste a YouTube playlist URL into PLAYLIST_URL first.")
if MAX_RESOLUTION <= 0:
    raise ValueError("MAX_RESOLUTION must be greater than zero.")
if START_ITEM is not None and START_ITEM <= 0:
    raise ValueError("START_ITEM must be greater than zero.")
if END_ITEM is not None and END_ITEM <= 0:
    raise ValueError("END_ITEM must be greater than zero.")
if START_ITEM is not None and END_ITEM is not None and START_ITEM > END_ITEM:
    raise ValueError("START_ITEM cannot be greater than END_ITEM.")

OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
output_template = str(
    OUTPUT_FOLDER
    / "%(playlist_title|Single Videos)s"
    / "%(playlist_index)03d - %(title)s [%(id)s].%(ext)s"
)

options = {
    "outtmpl": output_template,
    "download_archive": str(OUTPUT_FOLDER / ".downloaded.txt"),
    "ignoreerrors": True,
    "continuedl": True,
    "noplaylist": False,
    "windowsfilenames": True,
    "writedescription": True,
    "writeinfojson": True,
    "writethumbnail": True,
    "retries": 10,
    "fragment_retries": 10,
}

if AUDIO_ONLY:
    options.update({
        "format": "bestaudio/best",
        "postprocessors": [
            {"key": "FFmpegExtractAudio", "preferredcodec": "mp3", "preferredquality": "2"},
            {"key": "FFmpegMetadata"},
            {"key": "EmbedThumbnail"},
        ],
    })
else:
    options.update({
        "format": (
            f"bestvideo[height<={MAX_RESOLUTION}]+bestaudio/"
            f"best[height<={MAX_RESOLUTION}]/best"
        ),
        "merge_output_format": "mp4",
        "embedmetadata": True,
        "embedthumbnail": True,
    })

if DOWNLOAD_SUBTITLES:
    options.update({
        "writesubtitles": True,
        "writeautomaticsub": True,
        "subtitleslangs": ["en", "en.*"],
        "subtitlesformat": "srt/best",
    })
if COOKIES_FROM_BROWSER:
    options["cookiesfrombrowser"] = (COOKIES_FROM_BROWSER,)
if START_ITEM is not None:
    options["playliststart"] = START_ITEM
if END_ITEM is not None:
    options["playlistend"] = END_ITEM

if shutil.which("ffmpeg") is None:
    print("Warning: ffmpeg was not found. High-quality merging or MP3 conversion may fail.")

print(f"Saving files in: {OUTPUT_FOLDER.resolve()}")
with yt_dlp.YoutubeDL(options) as downloader:
    result_code = downloader.download([PLAYLIST_URL])

if result_code == 0:
    print("\nPlaylist download complete.")
else:
    print("\nFinished with one or more skipped or failed items.")

## 4. See the downloaded files

In [ ]:
downloaded_files = sorted(
    path for path in OUTPUT_FOLDER.rglob("*")
    if path.is_file() and path.name != ".downloaded.txt"
)
print(f"Found {len(downloaded_files)} downloaded/supporting files:\n")
for path in downloaded_files:
    print(path)